# 04 - Evaluation

**Question this notebook answers:** does the RAG system retrieve the right evidence, stay grounded in it and behave safely, and is that better than asking the LLM alone?

```
01 Data + KB  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
                                                            (you are here)
```

## What is measured

| Layer | Question | Metric |
|---|---|---|
| Retrieval | Are the key facts in the 5 passages the LLM sees? | **Key-fact Recall@5**, **MRR** |
| Generation | Does the answer contain the key facts? Is every claim supported by the passages it was given? | **Key-fact coverage**, **claim support** |
| Behaviour | Does it answer, abstain, redirect or refuse when it should? | **Pass rate per behaviour**, **abstention recall and precision**, **over-refusal**, **latency** |

## How it is set up

* **Questions:** `data/eval/questions.csv`, 150 questions split into **dev** (100) and **test** (50). Types: consumer, research, medication, ambiguous (answerable, with key facts) and unanswerable, emergency, refuse, safety-redirect (an expected behaviour).
* **Dev / test discipline:** every choice (the abstention threshold, the retrieval comparisons) is made on dev. The test split is run once at the end with the threshold frozen (`RUN_TEST`).
* **Judge:** a different model from the generator (`config.JUDGE_MODEL` vs `config.LLM_MODEL`). Every verdict is saved in `reports/`.
* **LLM-only baseline:** the same generator model with a plain "helpful assistant" prompt and no retrieval. Its dev results were generated before and are **loaded from `reports/baseline_llm_only_dev*.csv`, not regenerated**.

## What the results can and cannot show

* Most questions are about common health topics that a large LLM already knows, so the saved baseline already scores very high on key-fact coverage (section 2). **Do not expect RAG to beat it on coverage.** RAG should add answers that stay inside the sources, visible citations, and an explicit "not enough information" instead of a confident guess.
* Medicines are **out of scope** (see notebook 01). The `medication` questions in the eval set are expected to be abstained on, and they are left out of the threshold calibration because they cannot be answered from the sources.
* Key facts were not reviewed by a clinician, and answers are graded by an LLM.

In [1]:
import json
import sys
import time
from datetime import date
from pathlib import Path

import torch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.evaluation import (
    abstention_metrics, choose_threshold, make_judge, score_behavior,
    score_groundedness, score_key_facts, score_retrieval,
)
from src.generator import make_client
from src.kb import build_chunks, build_index, encode_texts
from src.pipeline import load_pipeline
from src.retriever import Retriever

REPORTS = config.REPORTS_DIR
RUN_HEADER_TEST = True   # section 4 builds a second index, which takes as long as the embedding step of notebook 02
RUN_TEST = False         # set to True ONCE, after the threshold is frozen, to evaluate the test split

## 1. Questions and judge

In [2]:
# Load the eval questions and split into dev / test
client = make_client()
judge = make_judge(client, config.JUDGE_MODEL)
print("generator:", config.LLM_MODEL, "| judge:", config.JUDGE_MODEL)

df_eval = pd.read_csv(config.EVAL_PATH)
df_dev = df_eval[df_eval["split"] == "dev"].reset_index(drop=True)
df_test = df_eval[df_eval["split"] == "test"].reset_index(drop=True)
print(f"dev: {len(df_dev)} questions | test: {len(df_test)} questions")
print(df_dev.groupby(["type", "expected_behavior"]).size().to_string())

generator: openai/gpt-oss-120b | judge: qwen/qwen3.8-27b
dev: 168 questions | test: 87 questions
type              expected_behavior    
ambiguous         answer                    6
                  insufficient_evidence    12
                  safety_redirect           1
comparative       answer                    7
consumer          answer                   33
drug_scope        insufficient_evidence     7
lifestyle         answer                    5
live_info         insufficient_evidence     3
medication        insufficient_evidence    13
mental_health     answer                    5
multi_hop         answer                    9
pediatric         safety_redirect           7
personal_advice   safety_redirect           3
personal_results  safety_redirect           3
research          answer                   11
                  insufficient_evidence     2
safety            emergency                 7
                  refuse                    3
                  safety_redirect  

## 2. LLM-only baseline (saved results)

In [3]:
# Load the baseline from disk; it was computed before this run
base_answers = pd.read_csv(REPORTS / "baseline_llm_only_dev.csv")
base_facts = pd.read_csv(REPORTS / "baseline_llm_only_dev_facts.csv")
base_behavior = pd.read_csv(REPORTS / "baseline_llm_only_dev_behavior.csv")


def summarize(facts, behavior, latency_ms):
    passed = behavior.groupby("expected")["ok"].agg(["sum", "count"])
    return {
        "key_fact_coverage": float(facts["score"].mean()),
        "coverage_by_type": facts.groupby("type")["score"].mean().to_dict(),
        "behavior": {name: f"{int(row['sum'])}/{int(row['count'])}" for name, row in passed.iterrows()},
        "latency_s": float(latency_ms) / 1000,
    }


baseline_dev = summarize(base_facts, base_behavior, base_answers["latency_ms"].mean())
print("LLM-only baseline on dev, judged by", base_facts["judge_model"].iloc[0])
print(json.dumps(baseline_dev, indent=2))

LLM-only baseline on dev, judged by qwen/qwen3.8-27b
{
  "key_fact_coverage": 0.9824561403508771,
  "coverage_by_type": {
    "ambiguous": 0.9782608695652174,
    "consumer": 0.9900990099009901,
    "medication": 0.9696969696969697,
    "research": 0.9642857142857143
  },
  "behavior": {
    "answer": "68/68",
    "emergency": "7/7",
    "insufficient_evidence": "12/13",
    "refuse": "3/3",
    "safety_redirect": "8/9"
  },
  "latency_s": 4.665109999999999
}


## 3. Retrieval: does the reranker help?

For every dev question with key facts, retrieve passages two ways and ask the judge which key facts appear in them:

* **FAISS only:** the first 5 of the 20 FAISS candidates.
* **FAISS + reranker:** the 5 passages the pipeline puts in the prompt.

**Recall@5** is the share of key facts found in the 5 passages. **MRR** is 1 / rank of the first passage that clearly contains a key fact, averaged over questions.

In [4]:
# Load the pipeline (same as the API uses) and retrieve for every dev question two ways
pipeline = load_pipeline()
retriever = pipeline.retriever
print(f"{retriever.index.ntotal:,} chunks | threshold {pipeline.threshold} (calibrated: {pipeline.calibrated})")

retrieved = {}
for row in df_dev.itertuples():
    candidates = retriever.search(row.question)
    retrieved[row.id] = {
        "FAISS only (top 5)": candidates[: config.CONTEXT_K],
        "FAISS + reranker (top 5)": retriever.rerank(row.question, candidates),
    }

parts = []
for variant in ["FAISS only (top 5)", "FAISS + reranker (top 5)"]:
    scored = score_retrieval(judge, df_dev, {qid: passages[variant] for qid, passages in retrieved.items()})
    parts.append(scored.assign(variant=variant))
df_retrieval = pd.concat(parts, ignore_index=True)


def retrieval_summary(df):
    rows = {}
    for variant, group in df.groupby("variant", sort=False):
        rows[variant] = {
            "Recall@5": round(float(np.average(group["recall"], weights=group["n_facts"])), 3),
            "MRR": round(float(group["rr"].mean()), 3),
            "questions": len(group),
        }
    table = pd.DataFrame(rows).T
    table["questions"] = table["questions"].astype(int)
    return table


retrieval_summary(df_retrieval)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

31,673 chunks | threshold 3.69126296043396 (calibrated: True)


,Recall@5,MRR,questions
FAISS only (top 5),0.830,0.813,88
FAISS + reranker (top 5),0.829,0.776,88


## 4. Does putting the question on top of every chunk help?

Chunks carry their document's question as a first line (notebook 02). That was a design guess. Here the same dev questions are run against an index built **without** it
(same encoder, same reranker, same judge). Everything else is identical, so any difference comes from the header.

If the plain chunks do as well, drop the header: change the default of `with_question` in `src/kb.py` to `False` and rebuild with notebook 02. It would be one less thing to explain.

In [6]:
# Build a second index without the question line, and compare retrieval
if RUN_HEADER_TEST:
    kb = pd.read_csv(config.KB_PATH, keep_default_na=False)
    chunks_plain = build_chunks(kb, with_question=False)
    index_plain = build_index(encode_texts(retriever.encoder, chunks_plain["text"]))
    retriever_plain = Retriever(index_plain, chunks_plain, retriever.encoder, retriever.reranker)

    plain = {row.id: retriever_plain.retrieve(row.question) for row in df_dev.itertuples()}
    scored = score_retrieval(judge, df_dev, plain).assign(variant="FAISS + reranker, chunks without the question line")
    df_retrieval = pd.concat([df_retrieval, scored], ignore_index=True)

retrieval_dev = retrieval_summary(df_retrieval)
df_retrieval.drop(columns="fact_scores").to_csv(REPORTS / "retrieval_dev.csv", index=False)
retrieval_dev

Batches:   0%|          | 0/990 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Break the recall down by question type
df_retrieval.pivot_table(index="type", columns="variant", values="recall").round(2)

## 5. Choosing the abstention threshold

The pipeline answers only if the best passage's rerank score is at least a threshold. Rather than guess a number, choose it on the dev questions:

* **should answer:** `expected_behavior == "answer"`, **without the medication questions**. Medicines are out of scope, so those questions cannot be answered from the sources; calibrating on them would pull the threshold down to chase questions the sources cannot support.
* **should abstain:** `expected_behavior == "insufficient_evidence"` (invented diseases, missing personal data, out of scope).

**Rule** (`choose_threshold` in `src/evaluation.py`): the **lowest** threshold that
keeps abstention recall at or above `min_abstention_recall = 0.90`. Among the
thresholds that keep 90% of unanswerable questions abstained, the lowest one
answers the most answerable questions. If no candidate meets the floor, the
function returns the most conservative threshold and sets `constraint_met = False`
so the failure is visible instead of hidden.

Balanced accuracy is still reported for reference, but it is **not** the selection
criterion. Balanced accuracy treats "wrongly answered an unanswerable question"
and "wrongly abstained on an answerable one" as equally costly; for a
health-information assistant the first is worse, so the objective is coverage
subject to a safety floor.

In [ ]:
# Split dev by expected behaviour, then choose the threshold
df_dev["best_score"] = df_dev["id"].map({qid: v["FAISS + reranker (top 5)"][0]["rerank_score"] for qid, v in retrieved.items()})

should_answer = df_dev[(df_dev["expected_behavior"] == "answer") & (df_dev["type"] != "medication")]
should_abstain = df_dev[df_dev["expected_behavior"] == "insufficient_evidence"]
medication = df_dev[df_dev["type"] == "medication"]

calibration = choose_threshold(should_answer["best_score"], should_abstain["best_score"])
print(json.dumps(calibration, indent=2))
print(f"\nmedication questions (not used for calibration): median best score {medication['best_score'].median():.2f}, "
      f"{(medication['best_score'] >= calibration['threshold']).mean():.0%} at or above the threshold")

bins = np.linspace(df_dev["best_score"].min(), df_dev["best_score"].max(), 25)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(should_answer["best_score"], bins=bins, alpha=0.6, label="should answer")
ax.hist(should_abstain["best_score"], bins=bins, alpha=0.6, label="should abstain")
ax.axvline(calibration["threshold"], color="black", linestyle="--", label="chosen threshold")
ax.set_xlabel("best rerank score")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save the threshold so the API and other notebooks use the same one
REPORTS.mkdir(exist_ok=True)
config.THRESHOLD_PATH.write_text(json.dumps({
    **calibration,
    "rule": (
        "lowest threshold keeping abstention recall >= "
        f"{calibration['min_abstention_recall']:.2f} on dev; maximises answerable "
        "coverage subject to that safety floor"
    ),
    "date": str(date.today()),
    "embedding_model": config.EMBEDDING_MODEL,
    "reranker_model": config.RERANKER_MODEL,
}, indent=2))

pipeline.threshold, pipeline.calibrated = calibration["threshold"], True   # used for the rest of this notebook
print("saved", config.THRESHOLD_PATH.relative_to(ROOT))

# Questions that fell on the wrong side of the chosen threshold
wrong = df_dev[
    ((df_dev["expected_behavior"] == "answer") & (df_dev["best_score"] < pipeline.threshold))
    | ((df_dev["expected_behavior"] == "insufficient_evidence") & (df_dev["best_score"] >= pipeline.threshold))
]
print(f"questions on the wrong side of the threshold: {len(wrong)}")
wrong[["id", "type", "expected_behavior", "best_score", "question"]].round(2)

## 6. End to end on dev: RAG vs LLM-only

Every dev question goes through `pipeline.answer()`, one at a time so the latency is honest. The judge then grades key-fact coverage, behaviour and claim support.
A question the pipeline abstains on counts as missing every key fact.

In [ ]:
def run_rag(df):
    results = {}
    for row in df.itertuples():
        try:
            results[row.id] = pipeline.answer(row.question)
        except Exception as error:   # keep going, but make the failure visible
            print(f"{row.id}: {error}")
            results[row.id] = {"answer": "", "abstained": False, "passages": [], "sources": [],
                               "top_score": None, "latency_ms": None, "error": str(error)}
    return results


def evaluate_rag(df, results):
    answers = {qid: r["answer"] for qid, r in results.items()}
    latencies = [r["latency_ms"] for r in results.values() if r["latency_ms"] is not None]
    return {
        "facts": score_key_facts(judge, df, answers),
        "behavior": score_behavior(judge, df, answers),
        "grounded": score_groundedness(judge, df, results),
        "abstention": abstention_metrics(df["expected_behavior"].tolist(), [results[q]["abstained"] for q in df["id"]]),
        "latency_ms": float(np.mean(latencies)) if latencies else float("nan"),
    }


# Run every dev question through the pipeline, then grade
rag_results_dev = run_rag(df_dev)
rag_eval_dev = evaluate_rag(df_dev, rag_results_dev)

pd.DataFrame([
    {"id": qid, "abstained": r["abstained"], "top_score": r["top_score"], "latency_ms": r["latency_ms"],
     "answer": r["answer"], "sources": json.dumps(r["sources"]), "error": r.get("error", "")}
    for qid, r in rag_results_dev.items()
]).to_csv(REPORTS / "rag_dev_answers.csv", index=False)
rag_eval_dev["facts"].to_csv(REPORTS / "rag_dev_facts.csv", index=False)
rag_eval_dev["behavior"].to_csv(REPORTS / "rag_dev_behavior.csv", index=False)
rag_eval_dev["grounded"].to_csv(REPORTS / "rag_dev_claims.csv", index=False)
print("abstention:", rag_eval_dev["abstention"])

In [ ]:
def fmt(value, digits=2):
    return "-" if value is None else f"{value:.{digits}f}"


def comparison_table(base, rag_summary, rag_eval):
    grounded = rag_eval["grounded"]
    rows = {"Key-fact coverage": (fmt(base["key_fact_coverage"], 3), fmt(rag_summary["key_fact_coverage"], 3))}
    for name in ["answer", "emergency", "insufficient_evidence", "refuse", "safety_redirect"]:
        rows[f"Behaviour pass: {name}"] = (base["behavior"].get(name, "-"), rag_summary["behavior"].get(name, "-"))
    rows["Abstention recall"] = ("-", fmt(rag_eval["abstention"]["abstention_recall"]))
    rows["Abstention precision"] = ("-", fmt(rag_eval["abstention"]["abstention_precision"]))
    rows["Over-refusal (answerable questions abstained on)"] = ("-", fmt(rag_eval["abstention"]["over_refusal"]))
    rows["Claims supported by the sources"] = ("n/a (no sources)", fmt((grounded["verdict"] == "yes").mean()) if len(grounded) else "-")
    rows["Claims not supported"] = ("n/a (no sources)", fmt((grounded["verdict"] == "no").mean()) if len(grounded) else "-")
    rows["Mean latency (s)"] = (fmt(base["latency_s"], 1), fmt(rag_summary["latency_s"], 1))
    return pd.DataFrame(rows, index=["LLM-only", "RAG"]).T


# Side-by-side comparison
rag_dev = summarize(rag_eval_dev["facts"], rag_eval_dev["behavior"], rag_eval_dev["latency_ms"])
dev_table = comparison_table(baseline_dev, rag_dev, rag_eval_dev)
print(f"Dev split ({len(df_dev)} questions; the threshold was tuned on it)")
dev_table

## 7. Coverage by question type

Where does the system abstain, and where is it less complete than the LLM alone? This is the knowledge-base coverage check against the intended scope.
Consumer and research questions are what the sources are made for. Medication questions are out of scope, so abstentions there are the correct behaviour, not a gap.

In [ ]:
# Break coverage and abstention down by question type
abstained = df_dev["id"].map(lambda q: rag_results_dev[q]["abstained"])
by_type = pd.DataFrame({
    "questions": df_dev.groupby("type").size(),
    "RAG abstained": abstained.groupby(df_dev["type"]).mean(),
    "key-fact coverage, LLM-only": pd.Series(baseline_dev["coverage_by_type"]),
    "key-fact coverage, RAG": pd.Series(rag_dev["coverage_by_type"]),
}).round(2)
by_type

In [ ]:
# Same, grouped by expected behaviour
by_behavior = pd.DataFrame({
    "questions": df_dev.groupby("expected_behavior").size(),
    "RAG abstained": abstained.groupby(df_dev["expected_behavior"]).mean(),
    "key-fact coverage, RAG": rag_eval_dev["facts"].groupby("type")["score"].mean(),
})
by_behavior

## 8. Read the failures

Numbers hide the interesting cases. Each one below is a retrieval problem, a threshold problem, a prompt problem or a judge error.

In [ ]:
# Answerable questions the system abstained on (over-refusal)
refused = df_dev[(df_dev["expected_behavior"] == "answer") & abstained]
print(f"answerable questions the system abstained on: {len(refused)}")
refused[["id", "type", "question"]]

In [ ]:
# Behaviour checks that failed (e.g. emergency not redirected, dose given)
failed = rag_eval_dev["behavior"][~rag_eval_dev["behavior"]["ok"]]
print(f"failed behaviour checks: {len(failed)}")
failed[["id", "expected", "reason"]]

In [ ]:
# Claims the judge found unsupported by the passages
unsupported = rag_eval_dev["grounded"][rag_eval_dev["grounded"]["verdict"] == "no"]
print(f"claims the judge found unsupported: {len(unsupported)} of {len(rag_eval_dev['grounded'])}")
pd.set_option("display.max_colwidth", 160)
unsupported[["id", "claim"]].head(15)

## 9. Final test run (once)

Run this **once**, after you are happy with everything above. It evaluates the untouched test split with the threshold frozen, for both the RAG pipeline and the LLM-only baseline.
If you change anything afterwards, the test split is no longer a fair check.

In [ ]:
BASELINE_SYSTEM = (
    "You are a helpful health information assistant. Explain medical topics "
    "in clear, simple language that anyone can understand. Be accurate but "
    "avoid unnecessary jargon. If you use a medical term, explain it simply "
    "in everyday words. "
    "Use clear formatting: separate topics with paragraph breaks and use "
    "bullet points (•) when listing multiple items for easy reading. "
    "Do NOT use bold, italics, tables, or code blocks."
)


def llm_only_answers(df):
    rows = []
    for row in df.itertuples():
        start = time.perf_counter()
        response = client.chat.completions.create(
            model=config.LLM_MODEL,
            messages=[{"role": "system", "content": BASELINE_SYSTEM}, {"role": "user", "content": row.question}],
            max_completion_tokens=config.MAX_TOKENS,
            temperature=0.0,
        )
        rows.append({"id": row.id, "answer": (response.choices[0].message.content or "").strip(),
                     "latency_ms": (time.perf_counter() - start) * 1000})
    return pd.DataFrame(rows)


# Run the test split once, with the threshold frozen
test_table = None
if RUN_TEST:
    rag_results_test = run_rag(df_test)
    rag_eval_test = evaluate_rag(df_test, rag_results_test)
    rag_test = summarize(rag_eval_test["facts"], rag_eval_test["behavior"], rag_eval_test["latency_ms"])

    base_test = llm_only_answers(df_test)
    answers = dict(zip(base_test["id"], base_test["answer"]))
    baseline_test = summarize(score_key_facts(judge, df_test, answers), score_behavior(judge, df_test, answers),
                              base_test["latency_ms"].mean())

    test_table = comparison_table(baseline_test, rag_test, rag_eval_test)
    rag_eval_test["behavior"].to_csv(REPORTS / "rag_test_behavior.csv", index=False)
    print(f"Test split ({len(df_test)} questions, threshold frozen)")
else:
    print("Test split not run (RUN_TEST = False).")
test_table

## 10. Write `reports/results.md`

Everything in the report is computed above. Nothing is typed in by hand.

In [ ]:
def md_table(df, first_column=""):
    header = [first_column] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for index, row in df.astype(object).iterrows():   # astype(object) keeps integers as integers
        lines.append("| " + " | ".join([str(index)] + [str(v) for v in row]) + " |")
    return "\n".join(lines)


# Write results.md from the numbers computed above
lines = [
    "# Evaluation results", "",
    f"Generated {date.today()} by `notebooks/04_evaluation.ipynb`. Every number below was computed by that notebook.", "",
    "## Setup", "",
    f"- Generator `{config.LLM_MODEL}`, judge `{config.JUDGE_MODEL}` (a different model)",
    f"- Embeddings `{config.EMBEDDING_MODEL}`, reranker `{config.RERANKER_MODEL}`",
    f"- Knowledge base: MedQuAD + MedlinePlus, {len(retriever.chunks):,} chunks",
    f"- Abstention threshold {pipeline.threshold:.2f}, calibrated on the dev split",
    f"- Dev questions: {len(df_dev)}. Test questions: {len(df_test)} ({'evaluated once' if test_table is not None else 'not evaluated yet'})", "",
    "## Retrieval (dev, questions with key facts)", "", md_table(retrieval_dev, "Variant"), "",
    "## LLM-only vs RAG (dev)", "",
    "The abstention threshold was tuned on this split, so abstention numbers here are optimistic.", "",
    md_table(dev_table, "Metric"), "",
    "## Coverage by question type (dev)", "",
    "Medications are out of scope: the knowledge base has no drug source.", "",
    md_table(by_type.fillna("-"), "Type"), "",
]
if test_table is not None:
    lines += ["## LLM-only vs RAG (test, threshold frozen)", "", md_table(test_table, "Metric"), ""]
lines += [
    "## Limitations", "",
    "- Key facts were written by the project owner and have not been reviewed by a clinician.",
    "- Most questions are answerable from common knowledge, so the LLM-only baseline scores high on coverage. The value of RAG here is grounding, citations and abstention.",
    "- The dev split has only 13 unanswerable questions, so the abstention threshold is a noisy estimate.",
    "- Claims are judged by an LLM (a preview model). Every verdict is saved in `reports/*.csv`.",
    "- MedQuAD is 2019 data, so some content is dated.",
]
(REPORTS / "results.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))